In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import os


In [2]:
train_dir = "dataset/train"
val_dir   = "dataset/val"
test_dir  = "dataset/test"

IMG_SIZE = 224
BATCH_SIZE = 32


In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)
test_data = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)


Found 6800 images belonging to 2 classes.
Found 1700 images belonging to 2 classes.
Found 30 images belonging to 2 classes.


In [4]:
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

for layer in base_model.layers:
    layer.trainable = False


In [5]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)


In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,218,788 (16.09 MB)

 Trainable params: 166,657 (651.00 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [7]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=5)
]

In [8]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=callbacks
)

Epoch 1/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 65s 292ms/step - accuracy: 0.5040 - loss: 0.7374 - val_accuracy: 0.5000 - val_loss: 0.6922 - learning_rate: 1.0000e-04
Epoch 2/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 67s 314ms/step - accuracy: 0.5172 - loss: 0.7187 - val_accuracy: 0.5006 - val_loss: 0.6899 - learning_rate: 1.0000e-04
Epoch 3/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 67s 315ms/step - accuracy: 0.5171 - loss: 0.7067 - val_accuracy: 0.5547 - val_loss: 0.6851 - learning_rate: 1.0000e-04
Epoch 4/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 68s 317ms/step - accuracy: 0.5406 - loss: 0.6971 - val_accuracy: 0.5518 - val_loss: 0.6811 - learning_rate: 1.0000e-04
Epoch 5/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 72s 338ms/step - accuracy: 0.5329 - loss: 0.7007 - val_accuracy: 0.5012 - val_loss: 0.6812 - learning_rate: 1.0000e-04
Epoch 6/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 72s 335ms/step - accuracy: 0.5493 - loss: 0.6911 - val_accuracy: 0.5100 - val_loss: 0.6786 - learning_rate: 1.0000e-04
Epoch 7/25
213/213 ━━━━━━━━━━━━━━━━━━━━ 67s 31

In [9]:
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)


Epoch 1/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 75s 335ms/step - accuracy: 0.5174 - loss: 0.9476 - val_accuracy: 0.5000 - val_loss: 1.6982 - learning_rate: 1.0000e-05
Epoch 2/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 78s 367ms/step - accuracy: 0.5326 - loss: 0.8591 - val_accuracy: 0.5029 - val_loss: 0.7051 - learning_rate: 1.0000e-05
Epoch 3/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 92s 429ms/step - accuracy: 0.5534 - loss: 0.8010 - val_accuracy: 0.5112 - val_loss: 0.6506 - learning_rate: 1.0000e-05
Epoch 4/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 93s 437ms/step - accuracy: 0.5690 - loss: 0.7658 - val_accuracy: 0.6665 - val_loss: 0.6219 - learning_rate: 1.0000e-05
Epoch 5/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 98s 459ms/step - accuracy: 0.5832 - loss: 0.7347 - val_accuracy: 0.5324 - val_loss: 0.6488 - learning_rate: 1.0000e-05
Epoch 6/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 90s 424ms/step - accuracy: 0.5949 - loss: 0.7037 - val_accuracy: 0.5259 - val_loss: 0.6579 - learning_rate: 1.0000e-05
Epoch 7/10
213/213 ━━━━━━━━━━━━━━━━━━━━ 70s 32

In [13]:
loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step - accuracy: 0.5667 - loss: 0.6109
Test Accuracy: 0.5666666626930237


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

preds = model.predict(test_data)
y_pred = (preds > 0.5).astype(int)

print(classification_report(test_data.classes, y_pred))
print(confusion_matrix(test_data.classes, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 726ms/step
              precision    recall  f1-score   support

           0       1.00      0.13      0.24        15
           1       0.54      1.00      0.70        15

    accuracy                           0.57        30
   macro avg       0.77      0.57      0.47        30
weighted avg       0.77      0.57      0.47        30

[[ 2 13]
 [ 0 15]]


In [12]:
model.save("efficientnet_pneumonia.h5")